# EL5M_01_Notebook
Javier Parada

September 23, 2021

## Purpose
This notebook documents the procedure followed in the creation of the following two indicators:

| Indicator Code | Indicator Name |
|---|---|
| AG.LND.EL5M.ZS | 	Land area where elevation is below 5 meters (% of total land area)|
| EN.POP.EL5M.ZS | 	Population living in areas where elevation is below 5 meters (% of total population) |

## Methodology

* Create binary layer to identify areas below 5 meters.
* Loop over 216 countries
* Generate zonal statistics using geemap.zonal_statistics_by_group
* Create country-level aggregates
* Present results

## Improvements:
- How can we obtain better shapes for every country?
- 
- 

## Results

Visualize results with plotly:

https://javierparada.github.io/EL5M_Area_MERIT

<img src="figure.png" width=400 height=400 />

## Suggested next steps
State suggested next steps, based on results obtained in this notebook.

- Explore methodologies for EN.POP.EL5M.ZS

# Setup

## Library import
We import all the required Python libraries

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Options for pandas
pd.options.display.max_columns = 50
pd.options.display.max_rows = 50

# Visualizations
import plotly
import plotly.graph_objs as go
import plotly.offline as ply
plotly.offline.init_notebook_mode(connected=True)

import cufflinks as cf
cf.go_offline(connected=True)
cf.set_config_file(theme='white')

import matplotlib as plt

# Autoreload extension
if 'autoreload' not in get_ipython().extension_manager.loaded:
    %load_ext autoreload
    
%autoreload 2

In [2]:
# Additional libraries
import os
import glob
import random
import folium
import wbgapi as wb
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import ee
ee.Initialize()
import leafmap
#leafmap.update_package()
import geemap
#geemap.update_package()

# Define directory

In [3]:
out_dir = os.path.join(os.path.expanduser('~'), 'Documents/Websites/JavierParada.github.io/EL5M/')
print(out_dir)

/Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/


# Parameter definition
We set all relevant parameters for our notebook. By convention, parameters are uppercase, while all the 
other variables follow Python's guidelines.

# Data import
Create list with 216 countries

In [4]:
df = wb.data.DataFrame(['AG.LND.EL5M.ZS', 'EN.POP.EL5M.ZS'], wb.region.members('WLD'), time=2010, labels=True)
df = df.sort_values(by=['Country'], ascending=True).reset_index()
df["economy"]= df["economy"].replace("XKX", "KSV") # replace code for Kosovo
df = df[df["economy"]!="CHI"] # remove Channel Islands
countries = list(df["economy"])
print('{} countries in WDI'.format(len(countries)))
df

216 countries in WDI


,economy,Country,AG.LND.EL5M.ZS,EN.POP.EL5M.ZS
0,AFG,Afghanistan,NaN,NaN
1,ALB,Albania,4.861273,7.070961
2,DZA,Algeria,0.025006,0.757501
3,ASM,American Samoa,3.555492,3.271014
4,AND,Andorra,NaN,NaN
...,...,...,...,...
212,VIR,Virgin Islands (U.S.),6.638373,5.444845
213,PSE,West Bank and Gaza,0.094975,0.553855
214,YEM,"Yemen, Rep.",0.414368,1.581226
215,ZMB,Zambia,NaN,NaN


# Border import
Create shapes for 216 countries

In [5]:
# 2015 GAUL Subnational Boundaries (Modified for World Bank Offical Use)
GAUL_1_256 = 'shapes/GAUL_1_256.shp'
GAUL_2_3411 = 'shapes/GAUL_2_3411.shp'

GAUL_borders_1 = gpd.read_file(os.path.join(out_dir,GAUL_1_256))
# replace codes for Channel Islands
GAUL_borders_1["ISO3166_1_"]= GAUL_borders_1["ISO3166_1_"].replace("GGY", "CHI")
GAUL_borders_1["ISO3166_1_"]= GAUL_borders_1["ISO3166_1_"].replace("JEY", "CHI")

# GAUL_borders_2
GAUL_borders_2 = gpd.read_file(os.path.join(out_dir,GAUL_2_3411))

# Remove Caspian Sea
# 2528 147307 1554 3098 1723
GAUL_borders_2 = GAUL_borders_2[(GAUL_borders_2["ADM1_CODE"]!=2528) & (GAUL_borders_2["ADM1_CODE"]!=147307) & (GAUL_borders_2["ADM1_CODE"]!=1554) & (GAUL_borders_2["ADM1_CODE"]!=3098) & (GAUL_borders_2["ADM1_CODE"]!=1723)]

In [6]:
codes = GAUL_borders_1[["ADM0_NAME", "ISO3166_1_"]]
codes

,ADM0_NAME,ISO3166_1_
0,Mozambique,MOZ
1,Mauritius,MUS
2,Malawi,MWI
3,Rwanda,RWA
4,Somalia,SOM
...,...,...
251,Sint Maarten (Neth.),SXM
252,Saint-Barthélemy (Fr.),None
253,Saint-Martin (Fr.),MAF
254,Kosovo,KSV


In [7]:
# Create file with borders based on GAUL_borders_2
# Merge ISO3166 codes because they are not available on GAUL_borders_2
borders = GAUL_borders_2.merge(codes, left_on='ADM0_NAME', right_on='ADM0_NAME', suffixes=('_left', '_right'))
print(borders.crs)

epsg:4326


In [8]:
pd.unique(borders["ISO3166_1_"])

array(['BDI', 'ATF', 'IOT', 'COM', 'DJI', 'ERI', 'ETH', 'KEN', 'MDG',
       'MOZ', 'MUS', 'MWI', 'FRA', 'RWA', 'SOM', 'SSD', 'SYC', 'TZA',
       'UGA', 'ZMB', 'ZWE', 'COG', 'GNQ', 'GAB', 'TCD', 'STP', 'COD',
       'CMR', 'CAF', 'AGO', 'SDN', 'TUN', 'MAR', 'LBY', 'EGY', 'DZA',
       'SWZ', 'ZAF', 'NAM', 'LSO', 'BWA', 'GMB', 'GHA', 'MRT', 'NGA',
       'GNB', 'SHN', 'TGO', 'CPV', 'NER', 'BEN', 'BFA', 'SEN', 'GIN',
       'LBR', 'MLI', 'CIV', 'SLE', 'GRD', 'JAM', 'KNA', 'MSR', 'TCA',
       'TTO', 'DOM', 'DMA', 'LCA', None, 'VCT', 'VGB', 'CYM', 'CUB',
       'HTI', 'BRB', 'PRI', 'VIR', 'BHS', 'ATG', 'AIA', 'ABW', 'MEX',
       'PAN', 'NIC', 'SLV', 'HND', 'GTM', 'CRI', 'BLZ', 'USA', 'SPM',
       'GRL', 'CAN', 'BMU', 'GUY', 'URY', 'VEN', 'PER', 'PRY', 'SGS',
       'SUR', 'ECU', 'COL', 'CHL', 'BRA', 'BOL', 'ARG', 'HMD', 'BVT',
       'TKM', 'UZB', 'TJK', 'KGZ', 'KAZ', 'TWN', 'MNG', 'PRK', 'MAC',
       'KOR', 'JPN', 'HKG', 'CHN', 'MMR', 'KHM', 'VNM', 'LAO', 'MYS',
       'PHL', 'THA', 

# Create a dictOfShapes

In [9]:
dictOfShapes = {}
for i, (iso, name) in enumerate(zip(df["economy"], df["Country"])):
    print(i+1, iso, name)
    
    # Save country shapes
    #lsoas = borders[borders["ISO3166_1_"] == iso]
    #lsoas.to_file("/Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/shapes/217/{}.shp".format(iso))
    
    # Plot country shapes
    #f, ax = plt.subplots(1, figsize=(6,6))
    #rgb = (random.random(), random.random(), random.random())
    #ax = lsoas.plot(ax=ax, color=rgb, edgecolor='black')
    #plt.title('Country {}: {} ({})'.format(i+1,name,iso))
    #plt.show()
    
    # Import country shapes into EE
    dictOfShapes["{0}".format(iso)] = geemap.shp_to_ee("/Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/shapes/217 simplified/{}.shp".format(iso), encoding='latin-1')

1 AFG Afghanistan
2 ALB Albania
3 DZA Algeria
4 ASM American Samoa
5 AND Andorra
6 AGO Angola
7 ATG Antigua and Barbuda
8 ARG Argentina
9 ARM Armenia
10 ABW Aruba
11 AUS Australia
12 AUT Austria
13 AZE Azerbaijan
14 BHS Bahamas, The
15 BHR Bahrain
16 BGD Bangladesh
17 BRB Barbados
18 BLR Belarus
19 BEL Belgium
20 BLZ Belize
21 BEN Benin
22 BMU Bermuda
23 BTN Bhutan
24 BOL Bolivia
25 BIH Bosnia and Herzegovina
26 BWA Botswana
27 BRA Brazil
Could not convert the geojson to ee.Geometry()
Invalid GeoJSON geometry.
28 VGB British Virgin Islands
29 BRN Brunei Darussalam
30 BGR Bulgaria
31 BFA Burkina Faso
32 BDI Burundi
33 CPV Cabo Verde
34 KHM Cambodia
35 CMR Cameroon
36 CAN Canada
37 CYM Cayman Islands


/Users/javierparada/opt/anaconda3/envs/geopandas/lib/python3.8/site-packages/shapefile.py:363: UserWarning:

Shapefile shape has invalid polygon: found orphan hole (not contained by any of the exteriors); interpreting as exterior.



38 CAF Central African Republic
39 TCD Chad
40 CHL Chile
41 CHN China
42 COL Colombia
43 COM Comoros
44 COD Congo, Dem. Rep.
45 COG Congo, Rep.
46 CRI Costa Rica
47 CIV Cote d'Ivoire
48 HRV Croatia
49 CUB Cuba
50 CUW Curacao
51 CYP Cyprus
52 CZE Czech Republic
53 DNK Denmark
54 DJI Djibouti
55 DMA Dominica
56 DOM Dominican Republic
57 ECU Ecuador
58 EGY Egypt, Arab Rep.
59 SLV El Salvador
60 GNQ Equatorial Guinea
61 ERI Eritrea
62 EST Estonia
63 SWZ Eswatini
64 ETH Ethiopia
65 FRO Faroe Islands
66 FJI Fiji
67 FIN Finland
68 FRA France
69 PYF French Polynesia
70 GAB Gabon
71 GMB Gambia, The
72 GEO Georgia
73 DEU Germany
74 GHA Ghana
75 GIB Gibraltar
76 GRC Greece
77 GRL Greenland
78 GRD Grenada
79 GUM Guam
80 GTM Guatemala
81 GIN Guinea
82 GNB Guinea-Bissau
83 GUY Guyana
84 HTI Haiti
85 HND Honduras
86 HKG Hong Kong SAR, China
87 HUN Hungary
88 ISL Iceland
89 IND India
Shapefile Reader requires a shapefile or file-like object. (no dbf file found)
90 IDN Indonesia


/Users/javierparada/opt/anaconda3/envs/geopandas/lib/python3.8/site-packages/geemap/common.py:7039: UserWarning:

The projection file /Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/shapes/217 simplified/IND.prj could not be found. Assuming the dataset is in a geographic coordinate system (GCS).



91 IRN Iran, Islamic Rep.
92 IRQ Iraq
93 IRL Ireland
94 IMN Isle of Man
95 ISR Israel
96 ITA Italy
97 JAM Jamaica
98 JPN Japan
99 JOR Jordan
100 KAZ Kazakhstan
101 KEN Kenya
102 KIR Kiribati
103 PRK Korea, Dem. People's Rep.
104 KOR Korea, Rep.
105 KSV Kosovo
106 KWT Kuwait
107 KGZ Kyrgyz Republic
108 LAO Lao PDR
109 LVA Latvia
110 LBN Lebanon
111 LSO Lesotho
112 LBR Liberia
113 LBY Libya
114 LIE Liechtenstein
115 LTU Lithuania
116 LUX Luxembourg
117 MAC Macao SAR, China
118 MDG Madagascar
119 MWI Malawi
120 MYS Malaysia
121 MDV Maldives
122 MLI Mali
123 MLT Malta
124 MHL Marshall Islands
125 MRT Mauritania
126 MUS Mauritius
127 MEX Mexico
128 FSM Micronesia, Fed. Sts.
129 MDA Moldova
130 MCO Monaco
131 MNG Mongolia
132 MNE Montenegro
133 MAR Morocco
134 MOZ Mozambique
135 MMR Myanmar
136 NAM Namibia
137 NRU Nauru
138 NPL Nepal
139 NLD Netherlands
140 NCL New Caledonia
141 NZL New Zealand
142 NIC Nicaragua
143 NER Niger
144 NGA Nigeria
145 MKD North Macedonia
146 MNP Northern Mariana I

# Data processing
Handling digital elevation models (DEMs)

In [10]:
# Each country has been saved as a feature collection in dictOfShapes
dictOfShapes["JPN"]

In [11]:
Map = geemap.Map(center = (40, -100), zoom = 3)
Map 
#Map.addLayer(countries, {}, 'Countries')

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=(T…

In [12]:
# Example:
pickCountry = 'JPN'
dictOfShapes[pickCountry]
# Add shapefile
Map.addLayer(dictOfShapes[pickCountry], {}, pickCountry)
Map.centerObject(dictOfShapes[pickCountry])

## Digital Elevation Models

https://developers.google.com/earth-engine/datasets/catalog/CGIAR_SRTM90_V4

https://developers.google.com/earth-engine/datasets/catalog/MERIT_DEM_v1_0_3?hl=en

In [13]:
# Digital elevation models (DEMs) 
srtm = ee.Image('CGIAR/SRTM90_V4').select('elevation')
merit = ee.Image("MERIT/DEM/v1_0_3").select('dem')

#.clip(dictOfShapes[pickCountry])

# Set visualization parameters for elevation
vis_params = {
  'min': 0,
  'max': 4000,
  'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5']}

Map.addLayer(srtm, vis_params, 'SRTM Digital Elevation Data Version 4', True, opacity=1.0)
Map.addLayer(merit, vis_params, 'MERIT DEM', True, opacity=1.0)

In [14]:
Fuji = ee.FeatureCollection(ee.Geometry.Point([138.7308, 35.3632]))
Map.addLayer(Fuji, {}, "Mount Fuji")

In [15]:
borders_gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")
Map.addLayer(borders_gaul, {}, 'Country Boundaries')

In [16]:
# Create a binary layer using logical operations.
lowAreas2 = srtm.lte(5)
lowAreas = merit.lte(5)

#.And(elevation.gte(0))
Map.addLayer(lowAreas2.selfMask(), {}, 'Low Areas SRTM<= 5 meters')
Map.addLayer(lowAreas.selfMask(), {}, 'Low Areas MERIT<= 5 meters')

# Run zonal statistics by country

In [ ]:
skip = ['AUS', 'BRA', 'IND', 'RUS']
rerun = ['DOM', 'ERI', 'FRA', 'GAB', 'NGA', 'NLD', 'NZL', 'PAK', 'PLW', 'PYF', 'SLV', 'TZA']

#skip = ['MEX', 'AUS', 'BRA', 'ECU', 'CHL', 'CAN', 'UGA', 'ZAF', 'RUS', 'MDG','THA','IDN', 'IND', 'GNB', 'EST', 'USA']

for i, (iso, name) in enumerate(zip(df["economy"], df["Country"])):
    print(i+1, iso, name)
    nlcd_stats = os.path.join(out_dir, 'results/SUM_{}.csv'.format(iso))  
    if iso not in skip:
        try:
            geemap.zonal_statistics_by_group(lowAreas, dictOfShapes[iso], nlcd_stats, statistics_type='SUM', denominator=1000000, decimal_places=2)
        except:
            print("An exception occurred")
    else:
        print("Skipped")

## ☆ Australia ☆

In [28]:
iso = 'AUS'
nlcd_stats = os.path.join(out_dir, 'results/SUM_{}.csv'.format(iso))  
#fc = geemap.geopandas_to_ee(borders[borders["ISO3166_1_"] == iso])
fc = geemap.shp_to_ee("shapes/217 simplified/AUS.shp")
#fc = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'Australia'))
geemap.zonal_statistics_by_group(lowAreas, fc, nlcd_stats, statistics_type='SUM', denominator=1000000, decimal_places=2)

aus_file = pd.read_csv('results/SUM_AUS.csv')
aus_file["ADM0_NAME"] = "Australia"
aus_file.to_csv('results/SUM_AUS.csv', index=False, encoding ='latin-1')

Computing ... 
Generating URL ...
Please wait ...
Data downloaded to /Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/results/SUM_AUS.csv


## ☆ Brazil ☆

In [29]:
iso = 'BRA'
nlcd_stats = os.path.join(out_dir, 'results/SUM_{}.csv'.format(iso))  
fc = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
geemap.zonal_statistics_by_group(lowAreas, fc, nlcd_stats, statistics_type='SUM', denominator=1000000, decimal_places=2)

Computing ... 
Generating URL ...
Please wait ...
Data downloaded to /Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/results/SUM_BRA.csv


## ☆ India ☆

In [30]:
iso = 'IND'
nlcd_stats = os.path.join(out_dir, 'results/SUM_{}.csv'.format(iso))  
fc = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'India'))
geemap.zonal_statistics_by_group(lowAreas, fc, nlcd_stats, statistics_type='SUM', denominator=1000000, decimal_places=2)

Computing ... 
Generating URL ...
Please wait ...
Data downloaded to /Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/results/SUM_IND.csv


## ☆ Russia ☆

In [31]:
iso = 'RUS'
nlcd_stats = os.path.join(out_dir, 'results/SUM_{}.csv'.format(iso))  
fc = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM0_NAME', 'Russian Federation'))
geemap.zonal_statistics_by_group(lowAreas, fc, nlcd_stats, statistics_type='SUM', denominator=1000000, decimal_places=2)

Computing ... 
Generating URL ...
Please wait ...
Data downloaded to /Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/results/SUM_RUS.csv


# Append all results

In [32]:
# 216 countries
os.chdir("/Users/javierparada/Documents/Websites/JavierParada.github.io/EL5M/results/")
extension = 'csv'
all_filenames = sorted([i for i in glob.glob('*.{}'.format(extension))])
len(all_filenames)

216

In [33]:
# allCountries
allCountries = pd.concat([pd.read_csv(f) for f in all_filenames])

allCountries["Percent_0"] = allCountries["Class_0"]/allCountries["Class_sum"]
allCountries["Percent_1"] = 1 - allCountries["Percent_0"]
allCountries = allCountries.sort_values(by=['Percent_1'], ascending=False)
# Add Australia to results from Australia
allCountries.loc[allCountries["ADM0_NAME"]=="Australia", "ISO3166_1_"] = "AUS"
allCountries.loc[allCountries["ADM0_NAME"]=="Brazil", "ISO3166_1_"] = "BRA"
allCountries.loc[allCountries["ADM0_NAME"]=="India", "ISO3166_1_"] = "IND"
allCountries.loc[allCountries["ADM0_NAME"]=="Russian Federation", "ISO3166_1_"] = "RUS"

allCountries.to_csv("../allCountries.csv", index=False, encoding ='latin-1')

In [34]:
national = allCountries.groupby("ISO3166_1_").sum()
national["Result_1"] = national["Class_1"]/national["Class_sum"]*100
national = national.sort_values(by=['Result_1'], ascending=False)
national.reset_index(inplace=True)
national = national.merge(codes, left_on='ISO3166_1_', right_on='ISO3166_1_', suffixes=('_left', '_right'))
national[['ADM0_NAME', 'Result_1']].head(20)

,ADM0_NAME,Result_1
0,The Bahamas,67.392400
1,Kiribati,58.177324
2,Netherlands,52.846412
3,Cayman Islands (U.K.),44.696170
4,Maldives,41.632481
5,Turks and Caicos Islands (U.K.),37.812791
6,Bahrain,34.322632
7,Marshall Islands,33.432263
8,Gibraltar (U.K.),30.769231
9,British Virgin Islands (U.K.),26.180685


# Display results

In [35]:
fig = go.Figure(data=go.Choropleth(
    locations = national['ISO3166_1_'],
    z = national['Result_1'],
    text = national['ADM0_NAME'],
    colorscale = 'inferno',
    autocolorscale=False,
    reversescale=True,
    marker_line_color='darkgray',
    marker_line_width=0.5,
    colorbar_ticksuffix = '%',
    colorbar_title = 'AG.LND.EL5M.ZS<br>',
))

fig.update_layout(
    title_text='Land area where elevation is below 5 meters (% of total land area)',
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='equirectangular'
    ),
    annotations = [dict(
        x=0.55,
        y=0.1,
        xref='paper',
        yref='paper',
        text='Source: <a href="https://databank.worldbank.org/source/world-development-indicators">\
            WDI</a>',
        showarrow = False
    )]
)

fig.show()
fig.write_html("../../EL5M_Area_MERIT.html")

end

# References
We report here relevant references:
1. https://towardsdatascience.com/choropleth-maps-101-using-plotly-5daf85e7275d
2. author2, article2, journal2, year2, url2

In [36]:
# Notes
#![figure.png](attachment:figure.png)